# Notebook 1 — Data Understanding & Data Quality Assessment

**Project:** Telecom Customer Churn Prediction

**Phase:** 1 of 3

**Notebook Objective**

The objective of this notebook is to validate the raw telecom customer dataset before any exploratory analysis or modeling begins. This includes verifying dataset integrity, identifying data quality issues, documenting business assumptions, and producing a cleaned dataset that serves as the single source of truth for all downstream notebooks.

This notebook intentionally avoids exploratory analysis and modeling. Its purpose is to ensure that every subsequent business insight and machine learning result is based on verified, trustworthy data.

# Data Understanding

**Business context:** [Company] is losing an estimated **$139,130/month** in recurring revenue to customer churn (26.5% of the customer base). This notebook is the first step toward identifying at-risk customers early enough for the Retention team to act — see `docs/executive_summary.md` for the full business case and why this project optimizes for **recall over precision** (missing an at-risk customer is far costlier than a false alarm).

**Objective of this notebook:** load the raw dataset, understand its structure, and identify any data quality issues that need resolving before EDA (`2_EDA.ipynb`) and modeling (`3_preprocessing_and_modeling.ipynb`). Column-level detail is documented separately in `docs/data_dictionary.md` — this notebook focuses on verifying and cleaning, not re-listing, that reference.

## 1. Load the Dataset

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# -----------------------------
# Reproducibility
# -----------------------------

RANDOM_STATE = 42

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [3]:
PROJECT_PATH = Path.cwd().parent
DATA_PATH = PROJECT_PATH / "Dataset" / "churn_dataset.csv"
OUTPUT_PATH = PROJECT_PATH / "Dataset" / "cleaned_dataset.csv"

## Dataset Information

- Source: IBM Telco Customer Churn Dataset
- Observations: 7,043
- Features: 21
- Target Variable: Churn
- File Used: Dataset/churn_dataset.csv

This notebook verifies that the raw dataset matches the expected source before any transformations are performed.

In [4]:
df = pd.read_csv(DATA_PATH)

print("\nFirst 5 rows:")
display(df.head())


First 5 rows:


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 2. Check Dataset Shape and Structure

In [5]:
print("Shape of dataset:", df.shape)

Shape of dataset: (7043, 21)


**7,043 rows, 21 columns.** One row per customer (confirmed unique via `customerID` in Section 4 below). This matches the expected size for this dataset.

In [6]:
print("Column names:")
print(df.columns.tolist())

Column names:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


21 columns: 1 identifier (`customerID`), 18 customer/service attributes, and the target (`Churn`). Full description of each column is in `docs/data_dictionary.md`.

## 3. Check for Duplicate Rows

In [7]:
duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


In [8]:
if duplicate_count > 0:
    df = df.drop_duplicates()
    print("Duplicates removed. New shape:", df.shape)
else:
    print("No duplicates found -- no action needed.")

No duplicates found -- no action needed.


**No duplicate rows.** No cleaning action needed for this check.

## 4. Check Data Types

In [9]:
print("Dataset Info:")
df.info()


Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-n

In [10]:
print("Data Types:")
print(df.dtypes)

Data Types:
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object


**Data quality issue found:** `TotalCharges` is typed as `object` (string), not numeric, even though it represents a dollar amount. This must be fixed before it can be used in any analysis or model -- addressed in Section 6.

## 5. Drop Non-Predictive Identifier and Fix Types

In [11]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['SeniorCitizen'] = df['SeniorCitizen'].astype('category')

- `TotalCharges` coerced to numeric. Any value that can't convert becomes `NaN` -- checked in Section 6.
- `SeniorCitizen` cast to `category` for clarity during EDA (it's binary but stored as int; this is cosmetic only, see `docs/data_dictionary.md` for the full note on this inconsistency).

In [12]:
print("Unique Customer IDs:", df["customerID"].nunique())
print("Rows:", len(df))

assert df["customerID"].nunique() == len(df)

print("Every row represents one unique customer.")

Unique Customer IDs: 7043
Rows: 7043
Every row represents one unique customer.


In [13]:
df.drop('customerID', axis=1, inplace=True)

`customerID` dropped -- confirmed unique per row, so it carries no predictive signal and would only add noise (or a target leak if accidentally left in) to any model.

## 6. Check for Missing Values

In [14]:
print("Missing Values:")
print(df.isnull().sum())

Missing Values:
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64


**All columns show 0 missing values except `TotalCharges`, which has 11.** These weren't visible as missing before the numeric coercion in Section 5 -- the raw file stores them as a literal blank string `" "`, not `NaN`, so a naive `isnull()` check on the raw column would have missed them entirely.

In [15]:
missing_total = df[df["TotalCharges"].isna()]

print("Affected Customers:", len(missing_total))

print()

print("Tenure Distribution")
print(missing_total["tenure"].value_counts())

print()

print("Monthly Charges Summary")

display(missing_total["MonthlyCharges"].describe())

Affected Customers: 11

Tenure Distribution
tenure
0    11
Name: count, dtype: int64

Monthly Charges Summary


count   11.00
mean    41.42
std     23.83
min     19.70
25%     20.12
50%     25.75
75%     58.97
max     80.85
Name: MonthlyCharges, dtype: float64

All 11 rows with missing `TotalCharges` have **`tenure = 0`** -- these are brand-new customers who haven't been billed a cumulative total yet. This explains the missingness and justifies the fix below.

In [16]:
df['TotalCharges'] = df['TotalCharges'].fillna(0)

In [17]:
assert df["TotalCharges"].isna().sum() == 0

print("Missing values successfully removed.")

Missing values successfully removed.


Filled with **0**, not the column median. Since every affected row is a customer with zero tenure, 0 is the value that actually reflects reality -- a median would introduce fake, unexplainable charges for customers who have never been billed.

## 7. Target Variable Distribution

In [18]:
print("Churn Distribution (Count):")
print(df['Churn'].value_counts())

Churn Distribution (Count):
Churn
No     5174
Yes    1869
Name: count, dtype: int64


In [19]:
print("Churn Distribution (Percentage):")
print(df['Churn'].value_counts(normalize=True) * 100)

Churn Distribution (Percentage):
Churn
No    73.46
Yes   26.54
Name: proportion, dtype: float64


**73.46% No / 26.54% Yes.** Moderately imbalanced -- not severe enough to require resampling (SMOTE, undersampling, etc.), but worth accounting for via `class_weight='balanced'` in modeling (see `3_preprocessing_and_modeling.ipynb`). This 26.54% baseline is also the reference point for the $139,130/month revenue-at-risk figure in `docs/executive_summary.md`.

## 8. Validate Categorical Domains

In [21]:
for col in df.select_dtypes(include="object"):

    print("="*60)

    print(col)

    print(df[col].value_counts(dropna=False))

gender
gender
Male      3555
Female    3488
Name: count, dtype: int64
Partner
Partner
No     3641
Yes    3402
Name: count, dtype: int64
Dependents
Dependents
No     4933
Yes    2110
Name: count, dtype: int64
PhoneService
PhoneService
Yes    6361
No      682
Name: count, dtype: int64
MultipleLines
MultipleLines
No                  3390
Yes                 2971
No phone service     682
Name: count, dtype: int64
InternetService
InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64
OnlineSecurity
OnlineSecurity
No                     3498
Yes                    2019
No internet service    1526
Name: count, dtype: int64
OnlineBackup
OnlineBackup
No                     3088
Yes                    2429
No internet service    1526
Name: count, dtype: int64
DeviceProtection
DeviceProtection
No                     3095
Yes                    2422
No internet service    1526
Name: count, dtype: int64
TechSupport
TechSupport
No                     34

Several service columns (`OnlineSecurity`, `TechSupport`, `StreamingTV`, etc.) have a third category, `"No internet service"`, in addition to `Yes`/`No`. This is **structural, not missing data** -- it's a direct consequence of `InternetService = "No"`. Treating it as a missing value to be imputed would be incorrect; it's kept as its own category.

## 9. Numerical Feature Summary

No statistical outlier treatment is performed in this notebook.

Outlier analysis belongs to exploratory data analysis and is intentionally deferred to Notebook 2.

In [22]:
print("Statistical Summary:")
display(df.describe())

Statistical Summary:


,tenure,MonthlyCharges,TotalCharges
count,"7,043.00","7,043.00","7,043.00"
mean,32.37,64.76,"2,279.73"
std,24.56,30.09,"2,266.79"
min,0.00,18.25,0.00
25%,9.00,35.50,398.55
50%,29.00,70.35,"1,394.55"
75%,55.00,89.85,"3,786.60"
max,72.00,118.75,"8,684.80"


`tenure` ranges 0-72 months, `MonthlyCharges` ranges $18.25-$118.75. No implausible values (e.g., negative charges or tenure) -- consistent with a clean, well-formed dataset apart from the `TotalCharges` issue already resolved.

## 10. Save Cleaned Dataset

In [23]:
verification_df = pd.read_csv(OUTPUT_PATH)
print("Verification Shape:", verification_df.shape)
assert verification_df.shape == df.shape
print("Clean dataset verified successfully.")

Verification Shape: (7043, 20)
Clean dataset verified successfully.


# Summary of Data Understanding Phase

### Dataset Overview

- Raw dataset successfully loaded and validated.
- 7,043 customer records verified.
- 21 original columns inspected.
- Every row represents a unique customer.
- No duplicate records detected.

### Data Quality Assessment

The raw dataset is generally clean and suitable for analysis.

Only one genuine data quality issue was identified:

- `TotalCharges` was stored as text rather than numeric.
- Eleven records contained blank strings that became missing values after numeric conversion.
- Every affected customer had `tenure = 0`, confirming that these customers had not yet accumulated any billing history.
- Missing values were therefore replaced with **0**, representing the true business state rather than using statistical imputation.

### Dataset Preparation

The following preprocessing steps were completed:

- Converted `TotalCharges` to numeric.
- Filled the verified missing values.
- Removed `customerID`.
- Converted `SeniorCitizen` to categorical for exploratory analysis.
- Saved the cleaned dataset for downstream analysis.

### Deliverables Produced

- Clean dataset (`cleaned_dataset.csv`)
- Verified data quality findings
- Documented cleaning decisions
- Dataset ready for exploratory analysis

### Next Phase

The next notebook focuses on exploratory data analysis to identify customer characteristics, service usage patterns, and business factors most strongly associated with churn.